# Week 1 Day 2 assignment: the cheapest agent that scores 15 / 25

**Challenge:** with a goal of at least 15 / 25 correct *consistently*, using the agent, what is the cheapest model you can find?

**Approach**
1. Make the lab's LangChain agent work with any OpenRouter model, and make it wait and retry when the provider hits a rate limit.
2. Choose candidates from OpenRouter's live model list: they must support tool calling and cost less than Luna. The `:batch` variants are left out because they're async-only.
3. **Screen:** one run of the queue for each candidate. Drop anything that scores below 13.
4. **Confirm:** three runs for each model that passes, and always for the Luna baseline. A model qualifies only if its *worst* run is at least 15, since "consistently" means the minimum, not the average.
5. Rank the qualifiers by cost per run.

Then: a chart of cost against accuracy, where the tokens go, which mistakes each model makes and what they would cost the business, and whether a tighter prompt can make a cheaper model qualify.

A note on the ceiling: 5 of the 25 claims are `manual`. They depend on the claims manual or the customer's history, which the agent doesn't have, so about 20 / 25 is the best a model can do here.

The setup cells below come from `labs/week1day2.ipynb`.

In [ ]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor
import urllib.request
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openrouter import OpenRouter
from pydantic import BaseModel, Field

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

In [ ]:
DATA = Path("../../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [ ]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

## The lab's instructions, tools and scoring

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

In [ ]:
recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

In [ ]:
AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1

## Step 1: an agent for any model

`make_agent` builds the lab's agent on any model, with any system prompt. `run_agent` runs it over the queue `runs` times. Within a run the claims go in parallel, which is safe because each claim records under its own id.

Some providers cap requests per minute; newer OpenRouter accounts have a cap on Luna (`new-account-rpm`). So when a claim gets a `429`, `decide_one` waits and tries again, up to 6 times. Without this, a rate-limited model scores 0 / 25 even though it never made a wrong decision. Other errors, like a model that loops past the recursion limit, count as wrong, as does a claim with nothing recorded.

Each row also keeps the tokens from OpenRouter's usage report, so we can see where the money goes.

In [ ]:
def make_agent(model, effort=None, system_prompt=AGENT_SYSTEM_PROMPT):
    """Build the claims agent on any OpenRouter model."""
    kwargs = {"reasoning_effort": effort} if effort else {}
    llm = ChatOpenAI(model=model, base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=3, **kwargs)
    return create_agent(model=llm, tools=[get_policy, record_decision], system_prompt=system_prompt)


def usage_of(messages):
    """Add up OpenRouter's cost and token counts over the model calls in one agent run."""
    totals = {"cost": 0.0, "calls": 0, "input_tokens": 0, "cached_tokens": 0, "output_tokens": 0, "reasoning_tokens": 0}
    for message in messages:
        if message.type != "ai":
            continue
        usage = message.response_metadata.get("token_usage") or {}
        totals["cost"] += usage.get("cost") or 0.0
        totals["calls"] += 1
        totals["input_tokens"] += usage.get("prompt_tokens") or 0
        totals["cached_tokens"] += (usage.get("prompt_tokens_details") or {}).get("cached_tokens") or 0
        totals["output_tokens"] += usage.get("completion_tokens") or 0
        totals["reasoning_tokens"] += (usage.get("completion_tokens_details") or {}).get("reasoning_tokens") or 0
    return totals


def decide_one(agent, model, effort, run_number, claim, label="default", attempts=6):
    """Run the agent on one claim and score what it recorded, waiting out rate limits."""
    start = time.time()
    for attempt in range(attempts):
        recorded.pop(claim["claim_id"], None)
        usage, error = usage_of([]), ""
        try:
            result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]}, {"recursion_limit": 12})
            usage = usage_of(result["messages"])
        except Exception as exception:
            error = f"{type(exception).__name__}: {str(exception)[:160]}"
        if "429" not in error:
            break
        time.sleep(20 * (attempt + 1))
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": error})
    truth = labels[claim["claim_id"]]
    return {
        "model": model, "effort": effort, "prompt": label, "run": run_number, "claim_id": claim["claim_id"],
        **answer, "handler_decision": truth["decision"], "difficulty": truth["difficulty"], "right": is_right(answer, truth),
        **usage, "seconds": round(time.time() - start, 1), "error": error,
    }


def run_agent(model, effort=None, runs=1, first_run=1, workers=4, system_prompt=AGENT_SYSTEM_PROMPT, label="default"):
    """Run the agent on the whole queue `runs` times; return one row per claim per run."""
    agent = make_agent(model, effort, system_prompt)
    rows = []
    for run_number in range(first_run, first_run + runs):
        recorded.clear()
        start = time.time()
        with ThreadPoolExecutor(max_workers=workers) as pool:
            batch = list(pool.map(lambda claim: decide_one(agent, model, effort, run_number, claim, label), claims))
        right = sum(row["right"] for row in batch)
        cost = sum(row["cost"] for row in batch)
        errors = sum(bool(row["error"]) for row in batch)
        missing = sum(row["decision"] == "nothing recorded" for row in batch)
        print(f"{model} ({effort}, {label} prompt) run {run_number}: {right}/{len(batch)} right, ${cost:.4f}, "
              f"{time.time() - start:.0f}s wall, {missing} nothing recorded, {errors} errors")
        rows += batch
    return rows


def summarize(rows):
    """One row per model and prompt: worst, mean and best score across runs, and mean cost per run."""
    keys = ["model", "effort", "prompt"]
    per_run = pd.DataFrame(rows).groupby(keys + ["run"], dropna=False).agg(right=("right", "sum"), cost=("cost", "sum")).reset_index()
    return per_run.groupby(keys, dropna=False).agg(
        runs=("run", "count"), min_right=("right", "min"), mean_right=("right", "mean"), max_right=("right", "max"),
        cost_per_run=("cost", "mean"),
    ).sort_values("cost_per_run")

## Step 2: the candidates

These are the tool-capable models on OpenRouter with input or output cheaper than Luna (`openai/gpt-6-luna`), with Luna at `low` as the baseline. The prices come live from OpenRouter's models endpoint, in dollars per million tokens.

In [ ]:
BASELINE = ("openai/gpt-6-luna", "low")

CANDIDATES = [
    BASELINE,
    ("qwen/qwen3.7-flash", "low"),
    ("openai/gpt-oss-20b", "low"),
    ("inclusionai/ling-3.0-flash", None),
    ("inception/mercury-2.5", "low"),
    ("openai/gpt-5-nano", "low"),
    ("nvidia/nemotron-3.5-lightning", "low"),
    ("deepseek/deepseek-v4.1-flash", "low"),
    ("deepseek/deepseek-v4-flash-0731", "low"),
    ("mistralai/mistral-nemo", None),
    ("google/gemini-2.5-flash-lite", None),
]

catalog = {m["id"]: m for m in json.load(urllib.request.urlopen("https://openrouter.ai/api/v1/models"))["data"]}
pd.DataFrame([{
    "model": model,
    "input $/M": float(catalog[model]["pricing"]["prompt"]) * 1e6,
    "cached input $/M": float(catalog[model]["pricing"].get("input_cache_read") or 0) * 1e6,
    "output $/M": float(catalog[model]["pricing"]["completion"]) * 1e6,
    "tools": "tools" in catalog[model]["supported_parameters"],
} for model, _ in CANDIDATES])

## Step 3: screen with one run each

In [ ]:
screen_rows = []
for model, effort in CANDIDATES:
    screen_rows += run_agent(model, effort, runs=1)

summarize(screen_rows)

## Step 4: confirm with three runs

Every model that scored at least 13 in the screen gets two more runs, and so does the Luna baseline whatever it scored. The screening run counts as run 1, so each one ends up with three.

In [ ]:
screen = summarize(screen_rows).reset_index()
finalists = [(row.model, None if pd.isna(row.effort) else row.effort) for row in screen.itertuples() if row.min_right >= 13]
if BASELINE not in finalists:
    finalists.insert(0, BASELINE)
print("Finalists:", finalists)

confirm_rows = []
for model, effort in finalists:
    confirm_rows += run_agent(model, effort, runs=2, first_run=2)

all_rows = screen_rows + confirm_rows
results = summarize([row for row in all_rows if (row["model"], row["effort"]) in finalists])
results

## Step 5: the answer

In [ ]:
qualifiers = results[(results.runs >= 3) & (results.min_right >= 15)]
display(qualifiers)
if len(qualifiers):
    (model, effort, _), best = next(qualifiers.iterrows())
    print(f"Cheapest consistent model: {model} ({effort}): worst {best.min_right:.0f}/25 over {best.runs:.0f} runs, ${best.cost_per_run:.4f} a run")
else:
    print("No candidate scored 15 or more on every run")

## Cost against accuracy

Each point is a model: its cost for a run of the queue against its worst score. The models that were screened out have only one run, so their "worst" is that single run. The dashed line is the target of 15. The cheapest point on or above the line is the answer. Points to the right of it on or above the line are consistent, but cost more.

In [ ]:
points = summarize(all_rows).reset_index()
points["status"] = points.apply(
    lambda row: "screened out" if row.runs < 3 else ("qualifies" if row.min_right >= 15 else "inconsistent"), axis=1)
points["name"] = points.model.str.split("/").str[1]
fig = px.scatter(
    points, x="cost_per_run", y="min_right", color="status", text="name", log_x=True,
    hover_data=["mean_right", "max_right", "runs"],
    labels={"cost_per_run": "Cost per run of 25 claims (USD, log scale)", "min_right": "Worst run (claims right of 25)"},
    title="Cheapest agent that scores 15 / 25 consistently",
    color_discrete_map={"qualifies": "#06D6A0", "inconsistent": "#FFD166", "screened out": "#EF476F"},
)
fig.add_hline(y=15, line_dash="dash", annotation_text="target: 15 / 25")
fig.update_traces(textposition="top center", marker_size=11)
fig.show()

## Where the tokens go

Most of the candidates cost less per token than Luna, but that doesn't make them cheaper per claim. This shows the average per claim for each model: the model calls the agent made, input tokens (and how many were served from the cache), output tokens and the reasoning tokens within them, and the cost.

In [ ]:
tokens = pd.DataFrame(all_rows).groupby("model").agg(
    calls=("calls", "mean"), input_tokens=("input_tokens", "mean"), cached_tokens=("cached_tokens", "mean"),
    output_tokens=("output_tokens", "mean"), reasoning_tokens=("reasoning_tokens", "mean"), cost_per_claim=("cost", "mean"),
).sort_values("cost_per_claim")
tokens["cost_per_claim"] = tokens["cost_per_claim"].map(lambda cost: f"${cost:.6f}")
tokens.round(1)

## Which mistakes, and what they would cost

Not all misses are equal. As the lab's *Commercial implications* note says, a wrong approval pays out money the policy doesn't owe, a wrong decline lets a customer down and a wrong referral costs a handler's time. This counts each kind of miss on the decision itself, averaged per run of 25 claims:

- **wrong approval:** the model approved, and the handler declined or referred. This risks money.
- **wrong decline:** the model declined, and the handler approved or referred. This hurts the customer.
- **missed referral, approved or declined:** included in the two above, where the handler wanted a colleague to look.
- **unneeded referral:** the model referred, and the handler decided. This costs handler time but is safe.
- **wrong payout:** the decision was right, but the payout was more than $1 out.
- **nothing recorded:** no decision at all, so a handler has to pick the claim up.

A model that misses by referring is much safer to deploy than one that misses by approving.

In [ ]:
def mistake(row):
    model, handler = row["decision"], row["handler_decision"]
    if row["right"]:
        return "right"
    if model == "nothing recorded":
        return "nothing recorded"
    if model == handler:
        return "wrong payout"
    if model == "approve":
        return "wrong approval"
    if model == "decline":
        return "wrong decline"
    return "unneeded referral"


mistakes = pd.DataFrame(all_rows)
mistakes["kind"] = mistakes.apply(mistake, axis=1)
runs_per_model = mistakes.groupby("model")["run"].nunique()
by_kind = mistakes.pivot_table(index="model", columns="kind", values="claim_id", aggfunc="count", fill_value=0)
by_kind = by_kind.div(runs_per_model, axis=0).round(1)
columns = ["right", "wrong approval", "wrong decline", "unneeded referral", "wrong payout", "nothing recorded"]
by_kind[[c for c in columns if c in by_kind.columns]].sort_values("right", ascending=False)

Here is the full picture for the cheapest consistent model: the handler's decision against the model's, over all its runs.

In [ ]:
best_rows = mistakes[mistakes.model == model]
pd.crosstab(best_rows.handler_decision, best_rows.decision, rownames=["handler"], colnames=[f"{model.split('/')[1]}"])

## What it means for the business

An illustrative comparison, with assumptions you should change for a real customer: a claims handler costs $40 an hour fully loaded and spends 20 minutes on a routine claim. Every claim the agent gets wrong still needs a handler, so the fair comparison counts the agent's misses as handler time too.

In [ ]:
HANDLER_HOURLY = 40
MINUTES_PER_CLAIM = 20
handler_per_claim = HANDLER_HOURLY * MINUTES_PER_CLAIM / 60

agent_per_claim = best.cost_per_run / 25
accuracy = best.mean_right / 25
blended = agent_per_claim + (1 - accuracy) * handler_per_claim

print(f"Handler alone:            ${handler_per_claim:.2f} a claim")
print(f"Agent alone:              ${agent_per_claim:.5f} a claim ({handler_per_claim / agent_per_claim:,.0f}x cheaper)")
print(f"Agent + handler on misses: ${blended:.2f} a claim, if a handler re-does every miss ({accuracy:.0%} accuracy)")
print("The catch: that assumes we know which claims are misses. Finding them is the hard part, and why the wrong-approval count above matters.")

## Can a better prompt make a cheaper model qualify?

`inclusionai/ling-3.0-flash` is the model to try: it costs about the same as Luna or less, and in earlier runs it scored between 14 and 16, just under the target. Its most common miss after wrong decisions was the payout, so the prompt below adds three things to the lab's instructions:

1. a reminder to call `record_decision` exactly once
2. the payout steps from section 10 spelled out, with the deductible and limits taken from the policy record
3. a nudge to refer when the documents don't match the story, rather than guessing

Everything else is the same: the same tools, policy wording and scoring, three runs.

In [ ]:
TIGHTER_INSTRUCTIONS = INSTRUCTIONS + """
How to work:
1. Call get_policy with the claim's policy_id first.
2. Check the claim date against the trip dates and the purchase date, and check which benefits and add-ons the plan includes.
3. Check every document the wording requires for this kind of claim. If one is missing, or the documents and the story disagree, refer.
4. For an approval, work out the payout exactly as section 10 says, in order: the eligible amount of each item (the lower of the amount claimed and any per-item limit), minus anything a carrier or another party has paid, minus the deductible where it applies, capped at the benefit maximum for this plan. Use the numbers in the policy record and the wording, not round figures.
5. Call record_decision exactly once, with payout 0 for a decline or a referral. Then reply with one short sentence.
"""

TIGHTER_SYSTEM_PROMPT = f"""{TIGHTER_INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""

PROMPT_MODEL = ("inclusionai/ling-3.0-flash", None)
prompt_rows = run_agent(*PROMPT_MODEL, runs=3, system_prompt=TIGHTER_SYSTEM_PROMPT, label="tighter")

comparison = summarize([row for row in all_rows if (row["model"], row["effort"]) in (PROMPT_MODEL, BASELINE)] + prompt_rows)
comparison

And the kinds of miss before and after the tighter prompt, per run:

In [ ]:
prompt_mistakes = pd.DataFrame([row for row in all_rows if (row["model"], row["effort"]) == PROMPT_MODEL] + prompt_rows)
prompt_mistakes["kind"] = prompt_mistakes.apply(mistake, axis=1)
(prompt_mistakes.pivot_table(index="prompt", columns="kind", values="claim_id", aggfunc="count", fill_value=0)
    .div(prompt_mistakes.groupby("prompt")["run"].nunique(), axis=0).round(1))

## Errors

An error is a timeout, a model that loops past the recursion limit, or a rate limit that outlasted six retries. It means the claim scores as wrong. Here is how many each model had, with the first message, so you can tell a provider problem apart from a model that can't call tools.

In [ ]:
errors = pd.DataFrame(all_rows + prompt_rows)
errors = errors[errors.error != ""]
errors.groupby("model").agg(count=("error", "count"), first_error=("error", "first")) if len(errors) else print("No errors")

## Accuracy by difficulty

This is the share of claims right at each level of difficulty. The `manual` claims need the claims manual or the customer's history, which the agent doesn't have, so expect almost none to be right.

In [ ]:
pd.DataFrame(all_rows).pivot_table(index="model", columns="difficulty", values="right", aggfunc="mean", sort=False)[["routine", "hard", "manual"]].round(2)

## Conclusion

**The cheapest model that scored at least 15 / 25 on every run, using the agent, was the lab's own Luna at `low`: 17, 18 and 18 out of 25, for about $0.0071 a run of the queue**, or under three hundredths of a cent a claim. With 5 `manual` claims out of reach, 20 is about the ceiling, so Luna is close to it.

| Model | Worst / mean of 3 runs | Cost per run | Verdict |
|---|---|---|---|
| `inclusionai/ling-3.0-flash` | 14 / 15.7 | $0.0060 | Cheaper, but missed 15 on one run |
| `inclusionai/ling-3.0-flash`, tighter prompt | 14 / 14.7 | $0.0045 | Cheaper still, but worse |
| **`openai/gpt-6-luna` (low)** | **17 / 17.7** | **$0.0071** | **Cheapest consistent** |
| `qwen/qwen3.7-flash` (low) | 15 / 15.7 | $0.0180 | Consistent, 2.5x the cost |
| `deepseek/deepseek-v4-flash-0731` (low) | 15 / 16.0 | $0.0236 | Consistent, 3.3x the cost |
| `deepseek/deepseek-v4.1-flash` (low) | 17 / 17.7 | $0.0316 | As good as Luna, 4.5x the cost |
| `nvidia/nemotron-3.5-lightning` (low) | 16 / 16.0 | $0.0353 | Consistent, 5x the cost |

Five more models (`gpt-oss-20b`, `mercury-2.5`, `gpt-5-nano`, `gemini-2.5-flash-lite`, `mistral-nemo`) scored between 0 and 12 in the screen.

### What I learned

1. **The price per token isn't the price per claim.** Most candidates cost less per token than Luna, but the token table shows why they cost more per claim. Luna writes about 320 output tokens a claim, while Ling writes 2,500, Qwen 3,200 and Nemotron 4,600, almost all of it reasoning. Luna also gets 96% of its input from the cache. Cost has to be measured on the real task, not read off a price list.
2. **"Consistently" changes the answer.** Ling's best run (17 / 25 at $0.0054) beats Luna's worst, but it dropped to 14 on another run. A single run would have picked the wrong model.
3. **A tighter prompt didn't rescue the cheaper model.** Spelling out the payout steps cut Ling's wrong payouts (2.3 to 1.7 a run) and its cost (it reasoned less), but it referred more claims it should have decided, so it scored a little worse overall. Prompt changes trade one mistake for another, so measure the whole score, not just the mistake you were trying to fix.
4. **The cheap models fail in the risky direction.** No model wrongly declined a claim the handler approved. Nearly every miss is a **wrong approval**, most often approving a claim the handler would have referred to a colleague. That's the mistake that pays out money the policy may not owe. For Luna, half of the claims the handler referred (9 of 18, over three runs) were approved instead. A deployment would need a check on approvals, not only an accuracy number.
5. **Rate limits can look like a bad model.** Before I added retries, Luna scored 0 / 25 in a parallel run. Every error was a `429` from a per-minute cap on newer accounts (`new-account-rpm`), not a wrong decision. Read the errors before you judge a model.
6. **The ceiling is set by the data, not the model.** No model gets the `manual` claims right, because they need the claims manual and the customer's history. Raising that ceiling is about context, not about a bigger model.

### Business framing

At an illustrative $40 an hour and 20 minutes a claim, a handler costs about $13 a claim, and the agent about $0.0003. But the agent is right about 71% of the time, and we don't know *which* 71%. So the value depends on catching the wrong approvals, which is a design question as much as a model question.

*Notes:* scores vary by one to three claims from run to run. OpenRouter's prices change too: two of the DeepSeek prices moved while I was running these experiments. The total spend for all the experiments, including my earlier runs, was about $1.50.